# Notebook 47 - Do compressed detectors forget unseen attack families? The second corpus (CIC-IoMT-2024)

**Why this notebook.** Notebook 41 held out one attack family at a time on CICIoT2023 (DoS, reconnaissance, web), trained teachers without it, pruned and recovered them, and found that the pruned detectors raise the unseen family as often as the dense detector recovered the same way, and raise unseen reconnaissance more often. That is the paper's third contribution and it rests on one corpus. This notebook repeats the protocol on CIC-IoMT-2024 through the notebook 32 bridge, holding out DoS (four TCP/IP classes, whose neighbour DDoS stays in training), reconnaissance (four classes) and spoofing (one class, ARP spoofing, the smallest family).

**What it does.** GPU, about six hours in total, resumable by family (teachers are cached under `models/iomt/lofo`, git-ignored). For each (network, held-out family, seed): a teacher trained from scratch on the remaining classes (4 epochs), accepted or excluded by a rule fixed in stage 2; a dense reference (two further epochs); magnitude and Fisher students at 90% realised MAC reduction, scores computed on the teacher, prefixes calibrated by realised MACs, recovered for two epochs; every model recalibrated. Endpoints: the share of the held-out family's evaluation rows raised as any attack (primary), where its rows go instead, the attack mass, and the in-distribution audit on the remaining classes.

**Readings, fixed in stage 2 before any result.** The notebook 41 rule: a student at 90% raising the unseen family at least 5 points less often than the dense reference, with the interval above zero, at matched in-distribution risk, in at least two of the three families for both rules; plus the cross-corpus expectation that no cell meets it (the CICIoT2023 reading).

In [ ]:
# Stage 1 - bootstrap (GPU runtime)
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import os, sys, json, gc, time, copy, itertools, subprocess, importlib
from pathlib import Path
import numpy as np, pandas as pd, torch, torch.nn as nn
from scipy import stats
import matplotlib
import matplotlib.pyplot as plt

REPO = Path("/content/drive/MyDrive/SABER_Research/saber-ids")
IOMT_REPO = Path("/content/drive/MyDrive/IOMT_Compression_Research/iomt-compression-research")
os.chdir(REPO)
for p in (str(REPO), str(IOMT_REPO / "src")):
    if p not in sys.path:
        sys.path.insert(0, p)
importlib.invalidate_caches()
from src.saber.bridge_iomt import build_bridge, iomt_taxonomy
from src.saber.taxonomy import DEFAULT_COST_PROFILES
from src.saber.metrics import full_model_audit
from src.saber.surgery import prune_cnn1d_channels, profile_forward_flops, count_parameters, enumerate_cnn1d_channel_groups
from src.saber.leverage import magnitude_scores, gradient_saliency_scores
from src.saber.postg5 import calibrate_prefix_count

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
R = REPO / "results/saber"; IN32 = R / "32_iomt_bridge"
OUT = R / "47_iomt_unseen_family"; OUT.mkdir(parents=True, exist_ok=True)
DATA_CACHE = REPO / "data/iomt_bridge"; TEACHER_DIR = REPO / "models/iomt/lofo"; TEACHER_DIR.mkdir(parents=True, exist_ok=True)   # git-ignored
assert (IN32 / "bridge_manifest.json").exists() and (IN32 / "teacher_audit.json").exists(), "NB32 results missing; pull the repo"
print("device:", DEVICE, "| torch", torch.__version__)

In [ ]:
# Stage 2 - pre-registration (frozen: rerunning this cell can never change it after its commit)
PREREG = {
    "notebook": "47_iomt_unseen_family",
    "question": ("on CIC-IoMT-2024, does a pruned and recovered detector raise an attack family it never saw in training less often "
                 "than the dense detector trained and recovered the same way, at matched in-distribution risk"),
    "motivation": ("notebook 41 answered this on CICIoT2023: no cell met the gap condition, unseen DoS and web attacks were raised on "
                   "99.7 to 100% of their flows by dense and pruned detectors alike, and unseen reconnaissance was raised more often by "
                   "the pruned students. A second corpus is the test of generality of the paper's third contribution"),
    "design": {
        "corpus": ("CIC-IoMT-2024 WiFi/MQTT through the notebook 32 bridge (official train side only, exact duplicates removed, seeded "
                   "20% stratified validation carve-out, 19 classes in 6 families, 45 features); evaluation subsample of notebooks 33 "
                   "to 45 (at most 4,000 rows per class, seed 0); the official test files are never opened"),
        "held_out_families": ["dos", "recon", "spoofing"],
        "held_out_note": ("DoS (four TCP/IP classes; its same-protocol neighbour DDoS stays in training, as DDoS did for DoS on "
                          "CICIoT2023), reconnaissance (four classes) and spoofing (one class, ARP spoofing, the smallest family, "
                          "in the role web application played on CICIoT2023); MQTT is not held out because it would remove five "
                          "classes and the only MQTT-protocol traffic"),
        "held_out_rule": ("every class of the held-out family is removed from the training rows, the class weights, the recalibration "
                          "slice and the Fisher batches; the models have one output per remaining class; the in-distribution audit uses "
                          "the evaluation rows of the remaining classes; the unseen audit uses the evaluation rows of the held-out "
                          "family; shared across settings and not refit per family: the bridge's per-feature clipping bounds and "
                          "standardisation (percentiles, mean and standard deviation of the full training split), which carry no "
                          "label information"),
        "architectures": "shallow CNN (64/128) and deep CNN (64/128/128/256) of notebooks 32 to 45, one output per remaining class",
        "teacher": ("trained from scratch per (architecture, family, seed): 4 epochs, Adam 1e-3, batch 1,024, class weights "
                    "count^-alpha normalised to unit mean with the notebook 32 exponent of the architecture (shallow 0.5, deep 0.35), "
                    "seeded initialisation and shuffle"),
        "teacher_acceptance": ("on the remaining classes' evaluation rows, after recalibration: benign escalation at most 0.15, attack "
                               "recall at least 0.85, fine macro-F1 at least 0.40; the escalation bound is 0.15 rather than notebook "
                               "41's 0.10 because the accepted notebook 32 shallow teacher of this corpus already escalates 0.087 on "
                               "the balanced subsample and teachers trained without a family vary by a few points across seeds; the "
                               "rule exists to exclude a broken teacher, not to select a good one; a cell whose teacher fails is "
                               "excluded from the verdict and listed"),
        "dense_reference": "the cell's teacher given two further epochs under the cell's recovery seed, then recalibrated (same treatment as the students)",
        "students": ("magnitude (L1) and Fisher scores computed on the teacher (Fisher on 40 training batches of remaining classes); "
                     "removal orders ascending with the per-layer floor of 8 channels used on this corpus; prefixes calibrated to 90% "
                     "realised MAC reduction (tolerance 1.5 points); recovered two full epochs over the remaining training rows under "
                     "the cell's recovery seed; recalibrated"),
        "recalibration": "BatchNorm running statistics re-estimated on the first 51,200 rows of a seed-2026 permutation of the remaining training rows, weights frozen",
        "seeds": [101, 211, 307],
        "cells": "2 architectures x 3 families x 3 seeds = 18 teachers, 18 dense references, 36 students",
    },
    "endpoints": {
        "unseen_detect": "fraction of the held-out family's evaluation rows predicted as any attack class (argmax rule), the primary endpoint",
        "unseen_absorption": "share of the held-out family's rows predicted as benign and as each remaining attack family (descriptive)",
        "unseen_attack_mass": "mean of 1 minus the benign probability on the held-out rows (threshold-free, descriptive)",
        "in_distribution": "recalibrated ground-truth HSR (balanced profile), benign escalation, attack-miss rate, family and fine macro-F1, ECE and per-family recall on the remaining classes",
        "paired_gap": "dense reference's unseen_detect minus the student's, under the same seed; mean over accepted seeds with a 95% t interval",
        "matched": "the mean paired difference in in-distribution HSR between student and dense reference is within 0.02",
    },
    "P1_gap": ("for an architecture, P1 holds if at 90% realised reduction, for at least two of the three held-out families and for "
               "both rules, the paired gap in unseen_detect is at least 0.05 with the lower end of its 95% interval above zero, and "
               "the cell is matched in distribution"),
    "readings": {
        "per_architecture": ("gap_at_matched_risk if P1 holds; gap_but_unmatched if the gap condition holds in at least two families for "
                             "both rules but matching fails in one of those cells; partial if at least one cell meets the gap condition "
                             "but P1 fails; no_gap if no cell meets it; a separate flag student_better is set when at least half of the "
                             "cells have a negative gap with the upper end of the interval below zero"),
        "overall": "gap_both / gap_one_architecture / no_gap (both read no_gap) / inconclusive",
    },
    "cross_corpus_expectation": "X4: both architectures read no_gap, as on CICIoT2023",
    "reporting_rule": "every cell reported as measured; a failed expectation changes the paper's generality statement, it is not dropped",
    "budgets": [0.90], "primary_budget": 0.90, "methods": ["magnitude", "fisher"],
    "gap_threshold": 0.05, "match_threshold": 0.02, "teacher_epochs": 4, "recovery_epochs": 2,
    "no_test_access": True,
}
pre_path = OUT / "N47_PREREGISTRATION.json"
if pre_path.exists():
    print("pre-registration already exists; left untouched:", pre_path)
else:
    pre_path.write_text(json.dumps(PREREG, indent=2)); print("pre-registration written:", pre_path)
print(json.dumps(PREREG, indent=2))

ARCHS = ["shallow", "deep"]; FAMILIES = PREREG["design"]["held_out_families"]; SEEDS = PREREG["design"]["seeds"]
METHODS = PREREG["methods"]; BUDGETS = PREREG["budgets"]; PRIMARY = PREREG["primary_budget"]
GAP = PREREG["gap_threshold"]; MATCH = PREREG["match_threshold"]; T_EPOCHS = PREREG["teacher_epochs"]; R_EPOCHS = PREREG["recovery_epochs"]
TOL = 0.015; CAL_SEED = 2026; FISHER_BATCHES = 40; MIN_W = 8
ACCEPT = {"b2a_max": 0.15, "attack_recall_min": 0.85, "fine_f1_min": 0.40}
ALPHA = {"shallow": 0.5, "deep": 0.35}
NAME = {"magnitude": "magnitude", "fisher": "Fisher", "dense": "dense"}
FAMN = {"benign": "benign", "ddos": "DDoS", "dos": "DoS", "mqtt": "MQTT", "recon": "recon", "spoofing": "spoofing"}

In [ ]:
# Stage 2b - commit the pre-registration before any result exists (skipped automatically once results exist)
def _prereg_committed():
    rel = "results/saber/47_iomt_unseen_family/N47_PREREGISTRATION.json"
    h = subprocess.run(["git", "log", "-1", "--format=%h", "--", rel], capture_output=True, text=True, cwd=REPO).stdout.strip()
    dirty = subprocess.run(["git", "status", "--porcelain", "--", rel], capture_output=True, text=True, cwd=REPO).stdout.strip()
    return bool(h) and not dirty


_results_exist = any(p.is_file() and p.name != "N47_PREREGISTRATION.json" for p in OUT.rglob("*"))
if _results_exist:
    assert _prereg_committed(), ("results exist but the pre-registration is not committed, so it can no longer be committed before any "
                                 "result; stop here and report it")
    print("results already exist in", OUT.name, "and the pre-registration is committed; nothing to do here")
else:
    os.chdir(REPO)
    r = subprocess.run(["python", "tools/commit_cell.py",
                        "NB47 pre-registration, committed before any result: leave-one-family-out on CIC-IoMT-2024 (NB32 bridge), the NB41 "
                        "protocol on the second corpus. Held out in turn: DoS, recon, spoofing; both networks; 3 seeds; teachers trained from "
                        "scratch without the family (4 epochs, acceptance rule E <= 0.15, attack recall >= 0.85, fine F1 >= 0.40, fixed before "
                        "any result), dense reference with two further epochs, magnitude and Fisher students at 90% realised MACs (scores on "
                        "the teacher, prefix calibrated) recovered for two epochs, every model recalibrated. Primary endpoint: share of the "
                        "unseen family raised as any attack, paired against the dense reference at matched in-distribution risk; P1 gap rule "
                        "as NB41 (>= 5 points, CI above zero, matched, two of three families, both rules). Expectation X4: no_gap on both "
                        "networks. No test access."],
                       capture_output=True, text=True)
    print(r.stdout); print(r.stderr)

In [ ]:
# Stage 3 - bridge and helpers (settings per held-out family)
TRAIN_LOADER, VAL_LOADER, CLASS_NAMES, FULL_TAX, MANIFEST = build_bridge(DATA_CACHE)
N_CLASSES = len(CLASS_NAMES); FAM_ALL = np.array(FULL_TAX.family_by_class); FAM_ORIG = list(MANIFEST["family_by_class"])
assert set(FAMILIES) <= set(FAM_ALL.tolist()), f"unknown family in {FAMILIES}; families are {sorted(set(FAM_ALL.tolist()))}"
Xt, Yt = TRAIN_LOADER.dataset.tensors; yt = Yt.numpy(); N_FEAT = int(Xt.shape[1])
Xv, Yv = VAL_LOADER.dataset.tensors; VAL_Y_ALL = Yv.numpy()
_rng = np.random.default_rng(0)
_idx = np.concatenate([_rng.permutation(np.where(VAL_Y_ALL == c)[0])[:4000] for c in range(N_CLASSES) if (VAL_Y_ALL == c).sum() > 0])
_idx = np.random.default_rng(12345).permutation(_idx)
EX_X_ALL = Xv[_idx]; EX_Y_ALL = VAL_Y_ALL[_idx]
del Xv, Yv
print("train rows", len(yt), "| eval rows", len(_idx), "| features", N_FEAT, "| families", sorted(set(FAM_ALL.tolist())))


class CNN1D(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        self.conv = nn.Sequential(nn.Conv1d(1, 64, 3, padding=1), nn.ReLU(), nn.BatchNorm1d(64),
                                  nn.Conv1d(64, 128, 3, padding=1), nn.ReLU(), nn.BatchNorm1d(128))
        self.pool = nn.AdaptiveAvgPool1d(1); self.head = nn.Linear(128, n_classes)

    def forward(self, x):
        if x.dim() == 2:
            x = x.unsqueeze(1)
        return self.head(self.pool(self.conv(x.float())).squeeze(-1))


class DeepCNN1D(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        def blk(i, o):
            return [nn.Conv1d(i, o, 3, padding=1), nn.ReLU(), nn.BatchNorm1d(o)]
        self.conv = nn.Sequential(*blk(1, 64), *blk(64, 128), nn.MaxPool1d(2), *blk(128, 128), *blk(128, 256))
        self.pool = nn.AdaptiveAvgPool1d(1); self.head = nn.Linear(256, n_classes)

    def forward(self, x):
        if x.dim() == 2:
            x = x.unsqueeze(1)
        return self.head(self.pool(self.conv(x.float())).squeeze(-1))


class Setting:
    '''Everything that depends on which family is held out: rows, labels, taxonomy, weights, calibration slice, evaluation rows.'''
    def __init__(self, family):
        self.family = family
        self.seen = [c for c in range(N_CLASSES) if FAM_ALL[c] != family]; self.unseen = [c for c in range(N_CLASSES) if FAM_ALL[c] == family]
        assert self.unseen, f"family {family} has no classes"
        self.remap = {c: i for i, c in enumerate(self.seen)}; self.names = [CLASS_NAMES[c] for c in self.seen]; self.K = len(self.seen)
        self.tax = iomt_taxonomy(self.names, [FAM_ORIG[c] for c in self.seen]); self.fam = np.array(self.tax.family_by_class); self.benign = self.names.index(self.tax.benign_class)
        self.seen_families = list(self.tax.families)
        keep = np.isin(yt, self.seen); self.train_idx = np.where(keep)[0]
        self.y_train = np.array([self.remap[c] for c in yt[keep]], dtype=np.int64); self.n_train = len(self.train_idx)
        self.ds = torch.utils.data.TensorDataset(Xt[torch.from_numpy(self.train_idx)], torch.from_numpy(self.y_train))
        counts = np.bincount(self.y_train, minlength=self.K); self.counts = counts
        self.class_w = {}
        for arch, alpha in ALPHA.items():
            w = np.zeros(self.K); w[counts > 0] = 1.0 / np.power(counts[counts > 0], alpha); w[counts > 0] /= w[counts > 0].mean()
            self.class_w[arch] = torch.tensor(w, dtype=torch.float32, device=DEVICE)
        cal = torch.randperm(self.n_train, generator=torch.Generator().manual_seed(CAL_SEED))[:50 * 1024]
        self.cal_loader = torch.utils.data.DataLoader(torch.utils.data.Subset(self.ds, cal.tolist()), batch_size=1024, shuffle=False)
        m_seen = np.isin(EX_Y_ALL, self.seen); m_un = np.isin(EX_Y_ALL, self.unseen)
        self.ex_x = EX_X_ALL[torch.from_numpy(m_seen)].to(DEVICE); self.ex_y = np.array([self.remap[c] for c in EX_Y_ALL[m_seen]], dtype=np.int64)
        self.un_x = EX_X_ALL[torch.from_numpy(m_un)].to(DEVICE); self.un_y = EX_Y_ALL[m_un]
        self.example = self.ex_x[:8].float(); self.fam_y = self.fam[self.ex_y]

    def loader(self, seed):
        return torch.utils.data.DataLoader(self.ds, batch_size=1024, shuffle=True, generator=torch.Generator().manual_seed(seed))

    def make(self, arch):
        return (CNN1D if arch == "shallow" else DeepCNN1D)(self.K)


def forward_logits(model, X):
    model.eval()
    with torch.no_grad():
        return torch.cat([model(X[i:i + 8192]).cpu() for i in range(0, len(X), 8192)]).numpy()


def train_epochs(model, loader, epochs, weights):
    opt = torch.optim.Adam(model.parameters(), lr=1e-3); lossf = nn.CrossEntropyLoss(weight=weights)
    for _ in range(epochs):
        model.train()
        for xb, yb in loader:
            opt.zero_grad(); lossf(model(xb.to(DEVICE)), yb.to(DEVICE)).backward(); opt.step()
    return model.eval()


def recalibrate_bn(model, loader, n_batches=50):
    for m in model.modules():
        if isinstance(m, nn.BatchNorm1d): m.reset_running_stats(); m.momentum = None
    model.train()
    with torch.no_grad():
        for i, (xb, _) in enumerate(loader):
            if i >= n_batches: break
            model(xb.to(DEVICE))
    for m in model.modules():
        if isinstance(m, nn.BatchNorm1d): m.momentum = 0.1
    model.eval(); return model


def softmax_np(z):
    z = z - z.max(axis=1, keepdims=True); e = np.exp(z); return e / e.sum(axis=1, keepdims=True)


def audit(model, S):
    '''In-distribution audit on the remaining classes, plus the unseen-family audit.'''
    lg = forward_logits(model, S.ex_x); a = full_model_audit(lg, S.ex_y, S.tax, DEFAULT_COST_PROFILES); pf = S.fam[lg.argmax(axis=1)]
    out = {"seen_hsr_balanced": float(a["hsr_balanced_soc"]), "seen_hsr_miss": float(a["hsr_miss_sensitive"]), "seen_hsr_fatigue": float(a["hsr_alert_fatigue"]),
           "seen_b2a": float(a["benign_to_attack_rate"]), "seen_a2b": float(a["attack_to_benign_rate"]), "seen_attack_recall": float(a["binary_attack_recall"]),
           "seen_family_f1": float(a["family_macro_f1"]), "seen_fine_f1": float(a["fine_macro_f1"]), "seen_ece": float(a["ece15"]),
           "seen_maxprob": float(softmax_np(lg).max(axis=1).mean())}
    out.update({f"seen_recall_{f}": float((pf[S.fam_y == f] == f).mean()) for f in S.seen_families})
    lu = forward_logits(model, S.un_x); pu = lu.argmax(axis=1); p = softmax_np(lu); fu = S.fam[pu]
    out.update({"unseen_rows": int(len(pu)), "unseen_detect": float((pu != S.benign).mean()), "unseen_attack_mass": float((1 - p[:, S.benign]).mean()),
                "unseen_maxprob": float(p.max(axis=1).mean())})
    out.update({f"absorbed_{f}": float((fu == f).mean()) for f in S.seen_families})
    return out


def accepted(row):
    return bool(row["seen_b2a"] <= ACCEPT["b2a_max"] and row["seen_attack_recall"] >= ACCEPT["attack_recall_min"] and row["seen_fine_f1"] >= ACCEPT["fine_f1_min"])


def valid_order(table, column, min_w):
    t = table.sort_values(column, ascending=True); remaining = t.groupby("module_path")["group_id"].count().to_dict(); order = []
    for row in t.itertuples():
        layer = str(row.module_path)
        if remaining[layer] - 1 < min_w: continue
        remaining[layer] -= 1; order.append((layer, int(row.channel_index)))
    return order


def prune_map_of(pairs):
    pm = {}
    for layer, ch in pairs: pm.setdefault(str(layer), []).append(int(ch))
    return {l: sorted(c) for l, c in pm.items()}


def widths(st):
    return "/".join(str(m.out_channels) for _, m in st.named_modules() if isinstance(m, nn.Conv1d))


SETTINGS = {f: Setting(f) for f in FAMILIES}
for f, S in SETTINGS.items():
    print(f"held out {f:9s}: classes {len(S.unseen)} | remaining classes {S.K} | train rows {S.n_train:,} | eval rows seen {len(S.ex_y):,} unseen {len(S.un_y):,} | remaining families {S.seen_families}")

In [ ]:
# Stage 4 - the runs: per (architecture, family, seed) one teacher, one dense reference, two students (resumable; teachers cached)
RUNS = OUT / "N47_runs.csv"
rows = pd.read_csv(RUNS).to_dict("records") if RUNS.exists() else []
done = {(r["architecture"], r["family"], int(r["seed"]), r["model"]) for r in rows}
MODELS = ["teacher_raw", "dense"] + [f"{m}_b{int(round(b * 100))}" for m in METHODS for b in BUDGETS]
struct_path = OUT / "N47_structures.csv"; struct_rows = pd.read_csv(struct_path).to_dict("records") if struct_path.exists() else []
(OUT / "structures").mkdir(exist_ok=True)
RUN_FAMILIES = FAMILIES          # narrow this list to split the run across sessions, e.g. ["dos"]; rerun with the full list to finish


def save_row(row):
    rows.append(row); done.add((row["architecture"], row["family"], int(row["seed"]), row["model"])); pd.DataFrame(rows).to_csv(RUNS, index=False)
    print(f"  {row['model']:14s}: unseen detect {row['unseen_detect']:.3f} (attack mass {row['unseen_attack_mass']:.3f}) | seen HSR {row['seen_hsr_balanced']:.4f} "
          f"E {row['seen_b2a']:.4f} M {row['seen_a2b']:.4f} famF1 {row['seen_family_f1']:.3f} | {row['seconds']:.0f}s")


def teacher_for(arch, S, seed):
    ck = TEACHER_DIR / f"{arch}_{S.family}_s{seed}.pt"; inner = S.make(arch)
    if ck.exists():
        inner.load_state_dict(torch.load(ck, map_location="cpu", weights_only=False)["state_dict"]); return inner.to(DEVICE).eval(), True
    torch.manual_seed(seed); np.random.seed(seed); model = S.make(arch).to(DEVICE); train_epochs(model, S.loader(seed), T_EPOCHS, S.class_w[arch])
    torch.save({"state_dict": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}, "architecture": arch, "family": S.family, "seed": seed, "classes": S.names}, ck)
    return model.eval(), False


for family, arch, seed in itertools.product(RUN_FAMILIES, ARCHS, SEEDS):
    if all((arch, family, seed, m) in done for m in MODELS): continue
    S = SETTINGS[family]; base = {"architecture": arch, "family": family, "seed": seed}
    print(f"\n== {arch} | held out {family} | seed {seed}")
    t0 = time.time(); teacher, cached = teacher_for(arch, S, seed)
    if (arch, family, seed, "teacher_raw") not in done:
        tr = copy.deepcopy(teacher); recalibrate_bn(tr, S.cal_loader); a = audit(tr, S)
        save_row({**base, "model": "teacher_raw", "method": "dense", "budget": 0.0, "parameters": int(count_parameters(teacher)), "realised_reduction": 0.0,
                  "teacher_accepted": accepted(a), "teacher_cached": cached, "seconds": round(time.time() - t0, 1), **a}); del tr
    teacher_row = next(r for r in rows if (r["architecture"], r["family"], int(r["seed"]), r["model"]) == (arch, family, seed, "teacher_raw"))
    acc = bool(teacher_row["teacher_accepted"])
    if not acc: print("  teacher FAILS the acceptance rule; its cell is excluded from the verdict and listed")
    if (arch, family, seed, "dense") not in done:
        t0 = time.time(); torch.manual_seed(seed + 1); dense = copy.deepcopy(teacher); train_epochs(dense, S.loader(seed + 1), R_EPOCHS, S.class_w[arch]); recalibrate_bn(dense, S.cal_loader)
        save_row({**base, "model": "dense", "method": "dense", "budget": 0.0, "parameters": int(count_parameters(dense)), "realised_reduction": 0.0,
                  "teacher_accepted": acc, "teacher_cached": cached, "seconds": round(time.time() - t0, 1), **audit(dense, S)}); del dense
    # scores on the teacher (remaining-class training batches), removal orders, calibrated structures
    groups = enumerate_cnn1d_channel_groups(teacher, S.example)
    mag = magnitude_scores(teacher, groups)
    grad = gradient_saliency_scores(teacher, S.loader(seed + 2), groups, device=DEVICE, max_batches=FISHER_BATCHES, criterion=nn.CrossEntropyLoss(weight=S.class_w[arch]))
    for t in (groups, mag, grad): t["group_id"] = t["group_id"].astype(str)
    table = groups[["group_id", "module_path", "channel_index"]].merge(mag, on="group_id", validate="1:1").merge(grad[["group_id", "fisher"]], on="group_id", validate="1:1")
    m0 = profile_forward_flops(teacher, S.example)["flops_per_item"]
    for method in METHODS:
        order = valid_order(table, method, MIN_W)
        build = lambda k, o=order: prune_cnn1d_channels(teacher, prune_map_of(o[:k]), S.example, minimum_remaining_per_layer=MIN_W)[0].to(DEVICE)
        realised = lambda st: 1.0 - profile_forward_flops(st, S.example)["flops_per_item"] / m0
        for budget in BUDGETS:
            name = f"{method}_b{int(round(budget * 100))}"
            if (arch, family, seed, name) in done: continue
            t0 = time.time(); cal = calibrate_prefix_count(order, target_reduction=budget, build_model=build, realised_reduction=realised, tolerance=TOL)
            assert cal.within_tolerance, f"{arch}/{family}/s{seed}/{name}: realised {cal.realised_reduction:.4f} is outside the tolerance"
            torch.manual_seed(seed + 1); st = build(cal.prefix_length)
            struct_rows = [r for r in struct_rows if (r["architecture"], r["family"], int(r["seed"]), r["model"]) != (arch, family, seed, name)]
            struct_rows.append({**base, "model": name, "k": cal.prefix_length, "realised_reduction": cal.realised_reduction, "widths": widths(st), "parameters": int(count_parameters(st))})
            pd.DataFrame(struct_rows).to_csv(struct_path, index=False)
            pd.DataFrame([{"module_path": l, "channel_index": c} for l, c in order[:cal.prefix_length]]).to_csv(OUT / f"structures/N47_{arch}_{family}_s{seed}_{name}_removed_groups.csv", index=False)
            train_epochs(st, S.loader(seed + 1), R_EPOCHS, S.class_w[arch]); recalibrate_bn(st, S.cal_loader)
            save_row({**base, "model": name, "method": method, "budget": budget, "parameters": int(count_parameters(st)), "realised_reduction": cal.realised_reduction,
                      "teacher_accepted": acc, "teacher_cached": cached, "seconds": round(time.time() - t0, 1), **audit(st, S)}); del st
    del teacher; gc.collect(); torch.cuda.empty_cache() if DEVICE == "cuda" else None
print("\nrows:", len(rows), "of", len(ARCHS) * len(FAMILIES) * len(SEEDS) * len(MODELS))

In [ ]:
# Stage 5 - analysis and verdict (rules exactly as pre-registered in stage 2)
runs = pd.read_csv(RUNS)
EXPECTED = len(ARCHS) * len(FAMILIES) * len(SEEDS) * len(MODELS)
assert len(runs) == EXPECTED, f"expected {EXPECTED} rows, found {len(runs)}: finish stage 4 (all families) first"
runs["teacher_accepted"] = runs["teacher_accepted"].astype(bool)
excluded = runs[(runs.model == "teacher_raw") & (~runs.teacher_accepted)][["architecture", "family", "seed", "seen_b2a", "seen_attack_recall", "seen_fine_f1"]]
ok = runs[runs.teacher_accepted]


def paired(arch, family, model, col):
    s = ok[(ok.architecture == arch) & (ok.family == family) & (ok.model == model)].set_index("seed")[col]
    d = ok[(ok.architecture == arch) & (ok.family == family) & (ok.model == "dense")].set_index("seed")[col]
    common = s.index.intersection(d.index); g = (d.loc[common] - s.loc[common]).to_numpy(); n = len(g)
    hw = stats.t.ppf(0.975, n - 1) * g.std(ddof=1) / np.sqrt(n) if n > 1 else np.nan
    return {"n": int(n), "mean": float(g.mean()) if n else np.nan, "lo": float(g.mean() - hw) if n > 1 else np.nan, "hi": float(g.mean() + hw) if n > 1 else np.nan}


cells = []
for arch, family, method, budget in itertools.product(ARCHS, FAMILIES, METHODS, BUDGETS):
    model = f"{method}_b{int(round(budget * 100))}"; S = SETTINGS[family]
    g = paired(arch, family, model, "unseen_detect"); h = paired(arch, family, model, "seen_hsr_balanced"); m = paired(arch, family, model, "unseen_attack_mass")
    d = ok[(ok.architecture == arch) & (ok.family == family) & (ok.model == "dense")]; s = ok[(ok.architecture == arch) & (ok.family == family) & (ok.model == model)]
    matched = bool(abs(h["mean"]) <= MATCH) if g["n"] else False
    cells.append({"architecture": arch, "family": family, "method": method, "budget": budget, "n_seeds": g["n"],
                  "dense_unseen_detect": float(d.unseen_detect.mean()), "student_unseen_detect": float(s.unseen_detect.mean()),
                  "gap": g["mean"], "gap_lo": g["lo"], "gap_hi": g["hi"], "gap_condition": bool(g["n"] > 1 and g["mean"] >= GAP and g["lo"] > 0),
                  "student_better": bool(g["n"] > 1 and g["mean"] < 0 and g["hi"] < 0),
                  "hsr_gap_student_minus_dense": -h["mean"] if g["n"] else np.nan, "matched": matched,
                  "dense_attack_mass": float(d.unseen_attack_mass.mean()), "student_attack_mass": float(s.unseen_attack_mass.mean()), "mass_gap": m["mean"],
                  "dense_to_benign": float(d[f"absorbed_benign"].mean()), "student_to_benign": float(s[f"absorbed_benign"].mean()),
                  **{f"student_absorbed_{f}": float(s[f"absorbed_{f}"].mean()) for f in S.seen_families}})
CELLS = pd.DataFrame(cells); CELLS.to_csv(OUT / "N47_cells.csv", index=False)

H = {}
for arch in ARCHS:
    c90 = CELLS[(CELLS.architecture == arch) & np.isclose(CELLS.budget, PRIMARY)]
    fam_ok = {f: bool(all(c90[(c90.family == f) & (c90.method == m)].gap_condition.iloc[0] for m in METHODS)) for f in FAMILIES}
    fam_ok_matched = {f: bool(all((c90[(c90.family == f) & (c90.method == m)].gap_condition.iloc[0] and c90[(c90.family == f) & (c90.method == m)].matched.iloc[0]) for m in METHODS)) for f in FAMILIES}
    P1 = bool(sum(fam_ok_matched.values()) >= 2); any_cell = bool(c90.gap_condition.any())
    reading = ("gap_at_matched_risk" if P1 else ("gap_but_unmatched" if sum(fam_ok.values()) >= 2 else ("partial" if any_cell else "no_gap")))
    H[arch] = {"P1": P1, "reading": reading, "families_meeting_gap_both_methods": fam_ok, "families_meeting_gap_both_methods_matched": fam_ok_matched,
               "student_better_flag": bool(c90.student_better.sum() >= len(c90) / 2), "cells_student_better": int(c90.student_better.sum()),
               "cells": c90[["family", "method", "n_seeds", "dense_unseen_detect", "student_unseen_detect", "gap", "gap_lo", "gap_hi", "gap_condition", "matched", "hsr_gap_student_minus_dense", "student_better"]].round(4).to_dict("records")}
readings = {a: H[a]["reading"] for a in ARCHS}
overall = ("gap_both" if all(r == "gap_at_matched_risk" for r in readings.values()) else
           "gap_one_architecture" if sum(r == "gap_at_matched_risk" for r in readings.values()) == 1 else
           "no_gap" if all(r == "no_gap" for r in readings.values()) else "inconclusive")
verdict = {"notebook": "47_iomt_unseen_family", "overall": overall, "X4_no_gap_both": bool(overall == "no_gap"), "by_architecture": H,
           "excluded_cells": excluded.round(4).to_dict("records"), "teachers_accepted": int(runs[runs.model == "teacher_raw"].teacher_accepted.sum()), "teachers": int((runs.model == "teacher_raw").sum()),
           "dense_unseen_detect": {f"{a}/{f}": round(float(ok[(ok.architecture == a) & (ok.family == f) & (ok.model == "dense")].unseen_detect.mean()), 4) for a in ARCHS for f in FAMILIES},
           "prereg": PREREG}
(OUT / "N47_verdict.json").write_text(json.dumps(verdict, indent=2, default=float))
print("OVERALL:", overall, "| X4 (no gap on both)", verdict["X4_no_gap_both"], "| teachers accepted", verdict["teachers_accepted"], "of", verdict["teachers"])
print("cells excluded (teacher failed acceptance):", len(excluded))
for arch in ARCHS:
    print(f"\n{arch}: reading {H[arch]['reading']} | P1 {H[arch]['P1']} | student better in {H[arch]['cells_student_better']} cells")
print(CELLS[["architecture", "family", "method", "n_seeds", "dense_unseen_detect", "student_unseen_detect", "gap", "gap_lo", "gap_hi", "matched", "hsr_gap_student_minus_dense", "student_to_benign"]].round(4).to_string(index=False))

In [ ]:
# Stage 6 - figure (600 dpi, paper names)
runs = pd.read_csv(RUNS); runs["teacher_accepted"] = runs["teacher_accepted"].astype(bool); ok = runs[runs.teacher_accepted]
plt.rcParams.update({"font.family": "serif", "font.serif": ["Liberation Serif", "DejaVu Serif"], "font.size": 7, "axes.labelsize": 7,
                     "xtick.labelsize": 6.5, "ytick.labelsize": 6.5, "legend.fontsize": 6, "pdf.fonttype": 42, "axes.linewidth": 0.6})
COLR = {"dense": "#1F1F1F", "magnitude_b90": "#E69F00", "fisher_b90": "#009E73"}; LAB = {"dense": "dense", "magnitude_b90": "magnitude 90%", "fisher_b90": "Fisher 90%"}
groups = [(a, f) for a in ARCHS for f in FAMILIES]; x0 = np.arange(len(groups)); w = 0.26
fig, ax = plt.subplots(figsize=(3.5, 1.8))
for k, mod in enumerate(["dense", "magnitude_b90", "fisher_b90"]):
    means, lo, hi = [], [], []
    for a, f in groups:
        v = ok[(ok.architecture == a) & (ok.family == f) & (ok.model == mod)].unseen_detect * 100
        means.append(v.mean()); lo.append(v.mean() - v.min()); hi.append(v.max() - v.mean())
    ax.bar(x0 + (k - 1) * w, means, w, color=COLR[mod], label=LAB[mod], yerr=[lo, hi], error_kw={"elinewidth": 0.6, "capsize": 1.2}, zorder=3)
ax.set_xticks(x0); ax.set_xticklabels([f"{FAMN[f]}\n{a}" for a, f in groups], fontsize=6)
ax.set_ylabel("unseen family raised as attack (%)"); ax.set_ylim(0, 128); ax.set_yticks([0, 25, 50, 75, 100])
ax.legend(frameon=False, loc="upper center", ncol=3, handlelength=1.0, columnspacing=0.8)
ax.spines[["top", "right"]].set_visible(False); ax.grid(axis="y", color="#E6E6E6", lw=0.5); ax.set_axisbelow(True)
fig.savefig(OUT / "N47_unseen.png", dpi=600, bbox_inches="tight"); fig.savefig(OUT / "N47_unseen.pdf", bbox_inches="tight"); plt.show()
print("written ->", OUT)

In [ ]:
# Stage 7 - commit the results (standard commit cell; the message is filled in from the verdict)
v = json.load(open(OUT / "N47_verdict.json"))
pre = subprocess.run(["git", "log", "-1", "--format=%h", "--", "results/saber/47_iomt_unseen_family/N47_PREREGISTRATION.json"], capture_output=True, text=True, cwd=REPO).stdout.strip() or "see log"
parts = []
for arch in ARCHS:
    h = v["by_architecture"][arch]
    cells_txt = "; ".join(f"{FAMN[c['family']]}/{NAME[c['method']]} dense {100 * c['dense_unseen_detect']:.1f}% student {100 * c['student_unseen_detect']:.1f}% (gap {100 * c['gap']:+.1f} [{100 * c['gap_lo']:+.1f}, {100 * c['gap_hi']:+.1f}] pts, matched {c['matched']})" for c in h["cells"])
    parts.append(f"{arch}: reading {h['reading']} (P1 {'HELD' if h['P1'] else 'FAILED'}); {cells_txt}.")
msg = (f"NB47 results (pre-registration {pre}): leave-one-family-out on CIC-IoMT-2024 (DoS, recon, spoofing; 3 seeds; teachers accepted "
       f"{v['teachers_accepted']} of {v['teachers']}, excluded cells {len(v['excluded_cells'])}), overall {v['overall']}; X4 (no gap on both networks) "
       f"{'HELD' if v['X4_no_gap_both'] else 'NOT HELD'}. " + " ".join(parts))
print(msg)
os.chdir(REPO)
r = subprocess.run(["python", "tools/commit_cell.py", msg], capture_output=True, text=True)
print(r.stdout); print(r.stderr)